# Classification with TabFORGE

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SilenceX12138/TabFORGE/blob/master/docs/tutorial/notebooks/prediction/01_classification.ipynb)

> **Colab setup:** Open this notebook with the badge, then follow the [tutorial setup guide](https://silencex12138.github.io/TabFORGE/#/tutorials.md) to install `tabforge-ml[tutorials]` from PyPI in this runtime. Select a GPU for pretrained workloads.

## 1. Overview

This tutorial fine-tunes `tabforge-v1` on a real-world classification dataset and reports balanced
accuracy alongside default TabPFN and random forest baselines.

In [ ]:
%pip install "tabforge-ml[tutorials]"

## 2. Imports and setup

The released pretrained checkpoints are used by default.

In [ ]:
import os
import tempfile
from pathlib import Path

from tabforge import (
    MODEL_ID,
    TabFORGEEmbeddingConfig,
    TabFORGERuntimeConfig,
    TabFORGETrainingConfig,
)

RANDOM_STATE = 7
MAX_TRAIN_ROWS = int(os.environ.get("TABFORGE_TUTORIAL_MAX_ROWS", "512"))
MAX_STEPS = int(os.environ.get("TABFORGE_TUTORIAL_MAX_STEPS", "1000"))
DEVICE = os.environ.get("TABFORGE_TUTORIAL_DEVICE", "auto")

assert MODEL_ID == "tabforge-v1"

## 3. Prepare a real-world dataset

`TabularDataset` downloads German Credit from OpenML, keeps the mixed numerical/categorical schema,
and performs a reproducible stratified split. The tutorial caps the training rows to keep execution
short; increase `TABFORGE_TUTORIAL_MAX_ROWS` for a larger run.

In [ ]:
from tabcamel.data.dataset import TabularDataset

dataset = TabularDataset("diabetes", task_type="classification")
split = dataset.split("stratified", train_size=0.8, random_state=RANDOM_STATE)
train_set = split["train_set"]
test_set = split["test_set"]

split_dict = train_set.split("stratified", train_size=0.8, random_state=RANDOM_STATE)
X_train = split_dict["train_set"].X_df.iloc[:MAX_TRAIN_ROWS].reset_index(drop=True)
y_train = split_dict["train_set"].y_s.iloc[:MAX_TRAIN_ROWS].reset_index(drop=True)
X_val = split_dict["test_set"].X_df.reset_index(drop=True)
y_val = split_dict["test_set"].y_s.reset_index(drop=True)
X_test = test_set.X_df.reset_index(drop=True)
y_test = test_set.y_s.reset_index(drop=True)
categorical_features = tuple(dataset.categorical_feature_list)

print({"train": X_train.shape, "val": X_val.shape, "test": X_test.shape, "target": y_train.name})

## 4. Configure TabFORGE

`n_prediction_samples` controls the number of independent target trajectories used to estimate class
probabilities.

In [ ]:
from tabforge import TabFORGEClassifier

classifier = TabFORGEClassifier(
    categorical_features=categorical_features,
    embedding_config=TabFORGEEmbeddingConfig(model_path="auto", n_folds=2),
    training_config=TabFORGETrainingConfig(max_steps=MAX_STEPS, batch_size=128),
    runtime_config=TabFORGERuntimeConfig(device=DEVICE),
    n_prediction_samples=32,
    random_state=RANDOM_STATE,
)

## 5. Fit

Validation data activates the validation-driven learning-rate schedulers.

In [ ]:
classifier.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    checkpoint="pretrained",
)

## 6. Predict

`predict_proba` returns probabilities in `classes_` order.

In [ ]:
probabilities = classifier.predict_proba(X_test)
predictions = classifier.predict(X_test)

print({"classes": classifier.classes_.tolist(), "probability_shape": probabilities.shape})

## 7. Compare with default classifiers

For context, fit standalone TabPFN and scikit-learn random forest classifiers with their default
parameters on exactly the same training split. Balanced accuracy is the mean recall across classes,
so it weights both classes equally.

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import balanced_accuracy_score
from tabpfn import TabPFNClassifier as ExternalTabPFNClassifier

tabpfn_classifier = ExternalTabPFNClassifier()
random_forest_classifier = RandomForestClassifier()

tabpfn_classifier.fit(X_train, y_train)
random_forest_classifier.fit(X_train, y_train)

comparison = pd.DataFrame(
    {
        "model": ["TabFORGE", "TabPFN", "Random forest"],
        "balanced_accuracy": [
            balanced_accuracy_score(y_test, predictions),
            balanced_accuracy_score(y_test, tabpfn_classifier.predict(X_test)),
            balanced_accuracy_score(y_test, random_forest_classifier.predict(X_test)),
        ],
    }
).set_index("model")
comparison

## 8. Save and load

Prediction checkpoints retain the fitted context needed for prediction.

In [ ]:
with tempfile.TemporaryDirectory() as directory:
    checkpoint_path = classifier.save_checkpoint(Path(directory))
    restored = TabFORGEClassifier.restore_from_checkpoint(checkpoint_path, device=DEVICE)
    restored_predictions = restored.predict(X_test.iloc[:8])

print({"checkpoint": checkpoint_path.name, "restored_predictions": restored_predictions.tolist()})

## 9. Optional advanced configuration

Use more prediction trajectories for smoother probabilities. For model selection, tune on validation
balanced accuracy and reserve the test split for one final report.